### Detecção de anomalias com Variational Auto Encoder
----


In [51]:
import pandas as pd
import sqlite3
import prettytable
from pathlib import Path
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

prettytable.DEFAULT = 'DEFAULT'

In [92]:
db_path = Path("KPIs") / "KPIs.db"
if not db_path.exists():
    raise FileNotFoundError(f"Database not found: {db_path.resolve()}")

conn = sqlite3.connect(db_path)
print("Connected to:", db_path.resolve())

Connected to: /home/tiago/Documentos/COURSERA_IBM_DATA_SCIENCE/TIM_KPIS_VAE/KPIs/KPIs.db


In [93]:
# Show available tables/views
sqlite_master = pd.read_sql_query(
    """
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY type, name
    """,
    conn,
)
sqlite_master

,name,type
0,KPI4G,table
1,KPI5G,table


### Modelo VAE
-----

In [94]:
# --- CONFIG ---
out_dir = "vae_output"
window_size = 3
latent_dim = 5
beta = 0.7
batch_size = 512
epochs = 80
random_state = 42
# --------------

os.makedirs(out_dir, exist_ok=True)

In [95]:
%%time
df = pd.read_sql_query("SELECT * FROM KPI4G", conn)
df['Dia'] = pd.to_datetime(df['Dia'], errors='coerce')
df = df.sort_values(['Site','Dia']).reset_index(drop=True)
# Ajuste de direção (menor = melhor → inverter)
df['PLOSS'] = -df['PLOSS'] # para garantir que todos os indicadores sejam do tipo "maior melhor"


CPU times: user 4.3 s, sys: 1.77 s, total: 6.07 s
Wall time: 6.1 s


In [96]:
%%time

# identificar colunas KPI (tudo após 'Site')
cols = list(df.columns)

if 'Site' not in cols:
    raise RuntimeError("Coluna 'Site' não encontrada.")

site_idx = cols.index('Site')
kpi_cols = cols[site_idx+1:]

print("KPIs:", kpi_cols)
# preencher NaN com mediana por Site (por KPI)
df[kpi_cols] = df[kpi_cols].apply(pd.to_numeric, errors='coerce')
df[kpi_cols] = df.groupby('Site')[kpi_cols].transform(
    lambda s: s.fillna(s.median())
)

# fallback: se algum Site tiver KPI totalmente NaN
df[kpi_cols] = df[kpi_cols].fillna(df[kpi_cols].median())

# dataframe apenas com colunas necessárias
df_kpi = df[['Dia','Site'] + kpi_cols].copy()

# converter para numérico
for c in kpi_cols:
    df_kpi[c] = pd.to_numeric(df_kpi[c], errors='coerce')

# ordenar apenas uma vez (importante)
df_kpi = df_kpi.sort_values(['Site','Dia'])


# =========================
# Construção janelas 3 dias
# =========================

X_list = []
idx_list = []

for site, g in df_kpi.groupby('Site'):
    vals = g[kpi_cols].values
    dates = g['Dia'].values
    n = vals.shape[0]

    if n < window_size:
        continue

    for i in range(n - window_size + 1):

        X_list.append(
            vals[i:i+window_size].reshape(-1))

        idx_list.append(
            (site, dates[i+window_size-1]))

X = np.vstack(X_list)
print("Amostras (janelas):", X.shape)

# =========================
# Train / Validation split
# =========================

X_train, X_val = train_test_split(
    X,
    test_size=0.2,
    random_state=random_state,
    shuffle=True)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

# =========================
# Normalização
# =========================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

joblib.dump(
    scaler,
    os.path.join(out_dir,"scaler.pkl"))

print("Scaler salvo")

KPIs: ['Disp', 'ACC_ERAB', 'DROP_ERAB', 'TPUT_USER_DL', 'TPUT_USER_UL', 'Usuarios', 'PRB_AVAIL', 'PRB_UTIL_DL', 'PRB_UTIL_UL', 'VOL_TOTAL_GB', 'PLOSS']
Amostras (janelas): (575804, 33)
Train: (460643, 33)
Val: (115161, 33)
Scaler salvo
CPU times: user 23.1 s, sys: 1.62 s, total: 24.7 s
Wall time: 24.8 s


In [97]:
%%time
input_dim = X_train_scaled.shape[1]
# ======================
# Encoder
# ======================

encoder_inputs = layers.Input(shape=(input_dim,))
x = layers.Dense(64, activation='relu')(encoder_inputs)
x = layers.Dense(32, activation='relu')(x)
x = layers.Dense(16, activation='relu')(x)
z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

@tf.keras.utils.register_keras_serializable()
def sampling(args):

    zm, zv = args
    eps = tf.random.normal(
        shape=(tf.shape(zm)[0], latent_dim)
    )
    return zm + tf.exp(0.5*zv)*eps

z = layers.Lambda(sampling)([z_mean,z_log_var])

encoder = Model(
    encoder_inputs,
    [z_mean,z_log_var,z],
    name="encoder")

# ======================
# Decoder
# ======================

latent_inputs = layers.Input(shape=(latent_dim,))
d = layers.Dense(16,activation='relu')(latent_inputs)
d = layers.Dense(32,activation='relu')(d)
d = layers.Dense(64,activation='relu')(d)

decoder_outputs = layers.Dense(input_dim)(d)

decoder = Model(
    latent_inputs,
    decoder_outputs,
    name="decoder")

# ======================
# VAE Custom Model
# ======================
class VAE(Model):

    def __init__(self, encoder, decoder, beta=1.0):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta


    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        return reconstruction

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            recon_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.square(data - reconstruction),
                    axis=1
                ))

            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var
                    - tf.square(z_mean)
                    - tf.exp(z_log_var),
                    axis=1
                ))
            loss = recon_loss + self.beta * kl_loss

        grads = tape.gradient(
            loss,
            self.trainable_weights
        )

        self.optimizer.apply_gradients(
            zip(grads, self.trainable_weights)
        )
        return {
            "loss": loss,
            "reconstruction_loss": recon_loss,
            "kl_loss": kl_loss
        }

vae = VAE(
    encoder,
    decoder,
    beta=beta
)

vae.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss = lambda y_true, y_pred: 0.0
)

encoder.summary()
decoder.summary()

Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_21      │ (None, 33)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_162 (Dense)   │ (None, 64)        │      2,176 │ input_layer_21[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_163 (Dense)   │ (None, 32)        │      2,080 │ dense_162[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_164 (Dense)   │ (None, 16)        │        528 │ dense_163[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_165 (Dense)   │ (None, 5)         │         85 │ dense_164[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_166 (Dense)   │ (None, 5)         │         85 │ dense_164[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_12 (Lambda)  │ (None, 5)         │          0 │ dense_165[0][0],  │
│                     │                   │            │ dense_166[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,954 (19.35 KB)

 Trainable params: 4,954 (19.35 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_22 (InputLayer)     │ (None, 5)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_167 (Dense)               │ (None, 16)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_168 (Dense)               │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_169 (Dense)               │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_170 (Dense)               │ (None, 33)             │         2,145 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,897 (19.13 KB)

 Trainable params: 4,897 (19.13 KB)

 Non-trainable params: 0 (0.00 B)

CPU times: user 176 ms, sys: 10 ms, total: 186 ms
Wall time: 183 ms


In [ ]:
%%time
# Hiperparâmetros (ajuste conforme necessário)
epochs_pretrain = 30           # curto - apenas para estimar erro
epochs_final = epochs          # pode usar sua variável epochs original
batch_size = batch_size        # usa variável que você já tem
k_mad = 4                     # multiplicador MAD (3 é razoável; 4 se quiser mais conservador)

# -------------------------
# EarlyStopping para pré-treino
# -------------------------
es_pre = EarlyStopping(
    monitor='loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

# -------------------------
# Pré-treino (modelo inicial, só para calcular erros)
# -------------------------
vae_pre = VAE(encoder, decoder, beta=beta)   # usa as definições já existentes
vae_pre.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=lambda y_true, y_pred: 0.0
)

vae_pre.fit(
    X_train_scaled,
    X_train_scaled,   # dummy target (seu train_step usa data[0])
    epochs=epochs_pretrain,
    batch_size=batch_size,
    validation_data=(X_val_scaled, X_val_scaled),
    callbacks=[es_pre],
    verbose=2
)

# -------------------------
# Erro de reconstrução por amostra (treino)
# -------------------------
X_train_pred = vae_pre.predict(X_train_scaled, batch_size=batch_size)
train_error = np.mean((X_train_scaled - X_train_pred)**2, axis=1)

# -------------------------
# Threshold robusto via MAD (ou troque para percentil se preferir)
# -------------------------
median = np.median(train_error)
mad = np.median(np.abs(train_error - median))
threshold = median + k_mad * mad

# Se preferir percentil 95:
# threshold = np.percentile(train_error, 95)

mask_train = train_error < threshold
X_clean = X_train_scaled[mask_train]

print(f"Treino: original={X_train_scaled.shape[0]}, clean={X_clean.shape[0]}, removidos={X_train_scaled.shape[0]-X_clean.shape[0]}")
if X_clean.shape[0] < X_train_scaled.shape[0] * 0.6:
    print("WARNING: >40% dos exemplos foram removidos — verifique threshold ou inspeção dos dados.")

# -------------------------
# (Opcional) validar threshold também em X_val — se ficar vazio, volta para X_val_scaled
# -------------------------
X_val_pred = vae_pre.predict(X_val_scaled, batch_size=batch_size)
val_error = np.mean((X_val_scaled - X_val_pred)**2, axis=1)
mask_val = val_error < threshold
X_val_clean = X_val_scaled[mask_val]
if X_val_clean.shape[0] < max(1, int(0.1 * X_val_scaled.shape[0])):  # se validação muito pequena, keep original
    X_val_clean = X_val_scaled
    print("Validação filtrada ficou muito pequena — mantendo X_val_scaled para validação final.")

# -------------------------
# Recriar modelos com pesos reinicializados
# -------------------------
# clonagem é prática conveniente: mantém arquitetura, zera pesos
encoder_clean = tf.keras.models.clone_model(encoder)
decoder_clean = tf.keras.models.clone_model(decoder)

# build para garantir shapes (útil para alguns backends)
encoder_clean.build((None, X_train_scaled.shape[1]))
decoder_clean.build((None, latent_dim))

vae_final = VAE(encoder_clean, decoder_clean, beta=beta)
vae_final.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=lambda y_true, y_pred: 0.0
)

# -------------------------
# EarlyStopping para treino final
# -------------------------
es_final = EarlyStopping(
    monitor='loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

# -------------------------
# Treino final no conjunto limpo
# -------------------------
history = vae_final.fit(
    X_clean,
    X_clean,
    epochs=epochs_final,
    batch_size=batch_size,
    validation_data=(X_val_clean, X_val_clean),
    callbacks=[es_final],
    verbose=2
)

Epoch 1/30
900/900 - 8s - 9ms/step - kl_loss: 6.0639 - loss: 12.5686 - reconstruction_loss: 8.3239 - val_loss: 0.0000e+00
Epoch 2/30
900/900 - 5s - 6ms/step - kl_loss: 5.5708 - loss: 9.8318 - reconstruction_loss: 5.9322 - val_loss: 0.0000e+00
Epoch 3/30
900/900 - 5s - 5ms/step - kl_loss: 5.6746 - loss: 11.1548 - reconstruction_loss: 7.1826 - val_loss: 0.0000e+00
Epoch 4/30
900/900 - 5s - 5ms/step - kl_loss: 5.3189 - loss: 8.6688 - reconstruction_loss: 4.9456 - val_loss: 0.0000e+00
Epoch 5/30
900/900 - 4s - 5ms/step - kl_loss: 5.3887 - loss: 8.6367 - reconstruction_loss: 4.8646 - val_loss: 0.0000e+00
Epoch 6/30
900/900 - 5s - 5ms/step - kl_loss: 5.4138 - loss: 8.5157 - reconstruction_loss: 4.7260 - val_loss: 0.0000e+00
Epoch 7/30
900/900 - 5s - 5ms/step - kl_loss: 5.5487 - loss: 9.2847 - reconstruction_loss: 5.4006 - val_loss: 0.0000e+00
Epoch 8/30
900/900 - 5s - 5ms/step - kl_loss: 5.5817 - loss: 9.1084 - reconstruction_loss: 5.2012 - val_loss: 0.0000e+00
Epoch 9/30
900/900 - 5s - 5ms/

In [ ]:
%%time
# salvar
# garantir que as próximas células usem o modelo treinado final
vae = vae_final

model_path = os.path.join(out_dir, "vae_model.keras")

try:
	# tenta salvar o modelo completo (pode falhar em modelos subclassed/custom)
	vae_final.save(model_path)
	print("Modelo completo salvo em:", model_path)
except Exception as e:
	# fallback robusto: salvar pesos
	model_path = os.path.join(out_dir, "vae_model.weights.h5")
	vae_final.save_weights(model_path)
	print("Aviso ao salvar modelo completo:", e)
	print("Pesos do modelo salvos em:", model_path)

joblib.dump({'latent_dim': latent_dim, 'beta': beta}, os.path.join(out_dir, 'config.pkl'))
print("Config salva em:", os.path.join(out_dir, 'config.pkl'))


Aviso ao salvar modelo completo: 
Object VAE was created by passing
non-serializable argument values in `__init__()`,
and therefore the object must override `get_config()` in
order to be serializable. Please implement `get_config()`.

Example:


class CustomLayer(keras.layers.Layer):
    def __init__(self, arg1, arg2, **kwargs):
        super().__init__(**kwargs)
        self.arg1 = arg1
        self.arg2 = arg2

    def get_config(self):
        config = super().get_config()
        config.update({
            "arg1": self.arg1,
            "arg2": self.arg2,
        })
        return config

Pesos do modelo salvos em: vae_output/vae_model.weights.h5
Config salva em: vae_output/config.pkl
CPU times: user 106 ms, sys: 14 ms, total: 120 ms
Wall time: 128 ms


In [ ]:
%%time
# 7) scores e threshold (99.5 percentile do treino)
X_train_pred = vae_final.predict(X_clean, verbose=0)
train_mse = np.mean(np.square(X_clean - X_train_pred), axis=1)
threshold = np.percentile(train_mse, 99.5)
print("Threshold (99.5 pctl treino limpo) =", threshold)

# 8) aplicar ao conjunto completo para obter score por janela e última janela por site
# idx_list contém (site, Dia) na mesma ordem de X
idx_df = pd.DataFrame(idx_list, columns=['Site','Dia'])
X_scaled_all = scaler.transform(X)
X_all_pred = vae_final.predict(X_scaled_all, verbose=0)
recon_error_matrix = np.square(X_scaled_all - X_all_pred)
scores_all = np.mean(recon_error_matrix, axis=1)
contrib_pct = recon_error_matrix / (recon_error_matrix.sum(axis=1, keepdims=True) + 1e-12)
idx_df['score'] = scores_all


#### ACHAR O KPI QUE MAIS CONTRIBUIU COM A ANOMALIA NA JANELA
n_kpis = len(kpi_cols)
n_days = recon_error_matrix.shape[1] // n_kpis
recon_reshaped = recon_error_matrix.reshape(
    -1,
    n_days,
    n_kpis)
kpi_error_total = recon_reshaped.sum(axis=1)
top_kpi_idx = np.argmax(kpi_error_total, axis=1)
top_kpis = [kpi_cols[i]for i in top_kpi_idx]
idx_df['top_kpi'] = top_kpis


# última janela por site
last_per_site = idx_df.sort_values('Dia').groupby('Site').tail(1).reset_index(drop=True)
site_stats = idx_df.groupby('Site')['score'].agg(['mean','std','count']).reset_index().rename(columns={'mean':'mu','std':'sigma','count':'n'})
last_per_site = last_per_site.merge(site_stats, on='Site', how='left')
last_per_site['zscore'] = (last_per_site['score'] - last_per_site['mu']) / (last_per_site['sigma'].replace(0, np.nan))
last_per_site['anomaly_global'] = last_per_site['score'] > threshold
last_per_site = last_per_site.sort_values('score', ascending=False)



Threshold (99.5 pctl treino limpo) = 2.8642057028516366
CPU times: user 27.2 s, sys: 3.02 s, total: 30.2 s
Wall time: 22.1 s


In [ ]:
%%time
# salvar CSVs
last_per_site.to_csv(os.path.join(out_dir, "last_per_site_scores4G.csv"), index=False)
idx_df.to_csv(os.path.join(out_dir, "all_window_scores4G.csv"), index=False)

# 9) mostrar top20
print("Top 20 anomalias (score):")
print(last_per_site[['Site','Dia','score','mu','sigma','zscore','anomaly_global','top_kpi']].head(20).to_string())


print("Artefatos gerados em:", out_dir)
print("Arquivos:", os.listdir(out_dir))

Top 20 anomalias (score):
           Site        Dia      score         mu      sigma    zscore  anomaly_global      top_kpi
347   5G-PIPIJ3 2026-02-26  43.415221  12.131448  12.512568  2.500188            True  Usuarios_SA
352   5G-PIPIJ1 2026-02-26  38.590686   4.013706   6.072549  5.693981            True  Usuarios_SA
283   4G-ITCE11 2026-02-26  29.960364   0.663841   3.971466  7.376752            True     ACC_FLOW
806   5G-TSIGJ1 2026-02-26  29.780268   2.191754   7.491991  3.682401            True     ACC_FLOW
1219  5G-CZCZJ2 2026-02-26  29.610558   1.196169   4.982547  5.702784            True     ACC_FLOW
414   5G-PZPZJ0 2026-02-26  19.647907   8.365970  19.338133  0.583404            True  Usuarios_SA
721   5G-NVFLJ0 2026-02-26   8.937554   0.424398   1.450264  5.870073            True  Usuarios_SA
2028  5D-FLPCB0 2026-02-26   8.647019   0.664356   1.743328  4.578978            True     ACC_NRRC
1574  5G-SASS10 2026-02-26   8.257559   5.968191   6.946776  0.329558            Tr